# Quantum-Enhanced RAG vs Classical RAG — NFCorpus

This notebook builds two retrieval pipelines on the same dataset (NFCorpus) so they can be fairly compared:

1. **Classical RAG**: Text → BGE-M3 embedding → FAISS cosine search
2. **Quantum RAG**: Text → BGE-M3 embedding → Autoencoder(1024→64) → 6-Qubit Variational Quantum Circuit (VQC) → FAISS cosine search

The only difference between the two pipelines is the Autoencoder + VQC step — everything else (documents, embedding model, chunking, distance metric) is kept identical, so that any difference in retrieval quality can be attributed to the quantum embedding step, not to some other confound.

**Evaluation metric**: Recall@5 — for a given question, did at least one of the top 5 retrieved documents match a document a human annotator actually marked as relevant (from NFCorpus's `qrels`)?


## Step 0 — Install dependencies

- `datasets` → load NFCorpus from HuggingFace
- `sentence-transformers` → run BGE-M3
- `faiss-cpu` → vector similarity search / indexing
- `pennylane` → build and train the quantum circuit (VQC)
- `scikit-learn` → PCA
- `torch` → training the VQC (PennyLane's torch interface plugs directly into PyTorch's autograd)


In [1]:
!pip install datasets sentence-transformers faiss-cpu pennylane scikit-learn numpy torch -q
!pip install -q beir 

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 77.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 76.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 937.5/937.5 kB 46.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.5/25.5 MB 72.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 75.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 167.2/167.2 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 111.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 20.8 MB/s eta 0:00:00


## Step 1 — Load the NFCorpus dataset

NFCorpus is a benchmark retrieval dataset (medical/nutrition abstracts) with three separate pieces:

- **`corpus`**: the actual documents. Each has `_id`, `title`, `text`.
- **`queries`**: real questions. Each has `_id`, `text`.
- **`qrels`**: the human-verified answer key. Each row is `{query-id, corpus-id, score}` — `score > 0` means a human confirmed that document genuinely answers that question. This is what makes a real, quantitative retrieval evaluation possible (Recall@k, etc.) instead of just eyeballing results.

Corpus and queries are independent lists — `qrels` is the only thing that links a specific question to its specific correct answer(s).


In [2]:
import os
os.environ["HF_HUB_OFFLINE"] = "0"
os.environ["HF_DATASETS_OFFLINE"] = "0"
from datasets import load_dataset

corpus = load_dataset("Hyukkyu/beir-nfcorpus", "corpus")["train"]
queries = load_dataset("Hyukkyu/beir-nfcorpus", "queries")["train"]
qrels = load_dataset("Hyukkyu/beir-nfcorpus-qrels")["train"]

print(f"Corpus size: {len(corpus)}")
print(f"Queries size: {len(queries)}")
print(f"Qrels rows: {len(qrels)}")
print(corpus[0])
print(queries[0])
print(qrels[0])

README.md: 0.00B [00:00, ?B/s]

corpus/train-00000-of-00001.parquet:   0%|          | 0.00/3.18M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3633 [00:00<?, ? examples/s]

queries/train-00000-of-00001.parquet:   0%|          | 0.00/80.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3237 [00:00<?, ? examples/s]

...

test.parquet:   0%|          | 0.00/35.5k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/110575 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/11385 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12334 [00:00<?, ? examples/s]

Corpus size: 3633
Queries size: 3237
Qrels rows: 110575
{'title': 'Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland', 'text': 'Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years afte

## Step 2 — Chunking

NFCorpus documents are already short (abstract-length, a few hundred words), well within BGE-M3's 8192-token limit. Splitting an already-short abstract into smaller pieces would only fragment coherent ideas and add unnecessary bookkeeping (which chunk came from which document, for qrels matching).

**Decision: 1 document = 1 chunk, no splitting.** Title and text are concatenated, since the title usually carries strong topical signal that helps the embedding model.


In [3]:
def build_chunks(corpus):
    """
    For NFCorpus, each document = one chunk (title + text combined).
    Returns a list of dicts: {chunk_id, doc_id, text}
    """
    chunks = []
    for doc in corpus:
        combined_text = (doc["title"] + ". " + doc["text"]).strip()
        chunks.append({
            "chunk_id": doc["_id"],   # same as doc_id since 1 doc = 1 chunk
            "doc_id": doc["_id"],
            "text": combined_text
        })
    return chunks

chunks = build_chunks(corpus)
print(chunks[0])
print(f"Total chunks: {len(chunks)}")

{'chunk_id': 'MED-10', 'doc_id': 'MED-10', 'text': 'Statin Use and Breast Cancer Survival: A Nationwide Cohort Study from Finland. Recent studies have suggested that statins, an established drug group in the prevention of cardiovascular mortality, could delay or prevent breast cancer recurrence but the effect on disease-specific mortality remains unclear. We evaluated risk of breast cancer death among statin users in a population-based cohort of breast cancer patients. The study cohort included all newly diagnosed breast cancer patients in Finland during 1995–2003 (31,236 cases), identified from the Finnish Cancer Registry. Information on statin use before and after the diagnosis was obtained from a national prescription database. We used the Cox proportional hazards regression method to estimate mortality among statin users with statin use as time-dependent variable. A total of 4,151 participants had used statins. During the median follow-up of 3.25 years after the diagnosis (range 0.

## Step 3 — Build shared lookup tables (once, up front)

Both `qrels` and `queries` only ever give you **IDs**, never the actual content directly matched up. Rather than rebuilding these lookups multiple times in different functions (a source of several bugs earlier in development), we build them **once, here, as standalone variables** that every later step can reuse:

- **`doc_id_to_chunk`**: document ID → the full chunk object (text, etc.)
- **`query_id_to_text`**: question ID → the actual question text
- **`query_to_pos_docs`**: question ID → list of document IDs a human confirmed are relevant to it (`score > 0` in qrels)
- **`all_doc_ids`**: every document ID that exists — used later to sample random "negative" (presumed irrelevant) documents
- **`query_ids_with_pos`**: only the questions that have at least one confirmed relevant document — you can only build training/eval examples from these


In [4]:
doc_id_to_chunk = {chunk["doc_id"]: chunk for chunk in chunks}
query_id_to_text = {q["_id"]: q["text"] for q in queries}

query_to_pos_docs = {}
for row in qrels:
    if row["score"] > 0:
        query_to_pos_docs.setdefault(row["query-id"], []).append(row["corpus-id"])

all_doc_ids = list(doc_id_to_chunk.keys())
query_ids_with_pos = list(query_to_pos_docs.keys())

print(f"doc_id_to_chunk: {len(doc_id_to_chunk)} entries")
print(f"query_id_to_text: {len(query_id_to_text)} entries")
print(f"query_to_pos_docs: {len(query_to_pos_docs)} queries with known relevant docs")
print(f"all_doc_ids: {len(all_doc_ids)} total documents")

doc_id_to_chunk: 3633 entries
query_id_to_text: 3237 entries
query_to_pos_docs: 2590 queries with known relevant docs
all_doc_ids: 3633 total documents


In [5]:
# Step 3.5 — Fix all seeds, then split queries into train/test BEFORE any training happens
import random
import numpy as np
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

query_ids_with_pos_shuffled = query_ids_with_pos.copy()
random.shuffle(query_ids_with_pos_shuffled)

n_test = 100
test_qids = query_ids_with_pos_shuffled[:n_test]
train_qids = query_ids_with_pos_shuffled[n_test:]

print(f"Total queries with positives: {len(query_ids_with_pos)}")
print(f"Train queries (for triplets): {len(train_qids)}")
print(f"Test queries (held out for eval): {len(test_qids)}")
assert set(train_qids).isdisjoint(set(test_qids)), "Leakage: train/test overlap!"

Total queries with positives: 2590
Train queries (for triplets): 2490
Test queries (held out for eval): 100


## Step 4 — Embed all chunks with BGE-M3

BGE-M3 converts each chunk of text into a dense 1024-dimensional vector. `normalize_embeddings=True` L2-normalizes every vector to unit length — this matters because it lets cosine similarity be computed as a plain inner product later (inner product on normalized vectors = cosine similarity), which is what FAISS's `IndexFlatIP` expects.

This step is also the **classical embedding** that the classical RAG baseline will use directly (no PCA, no VQC) — so this cell's output (`embeddings`) feeds *both* pipelines.


In [6]:
import os
import numpy as np
from sentence_transformers import SentenceTransformer

embedding_file = "/kaggle/working/corpus_embeddings.npy"

model = SentenceTransformer("BAAI/bge-m3")
# Check if pre-computed embeddings exist
if os.path.exists(embedding_file):
    print(f"Loading cached embeddings from '{embedding_file}'...")
    embeddings = np.load(embedding_file)
else:
    print("No cached embeddings found. Computing with BGE-M3...")
    
    texts = [chunk["text"] for chunk in chunks]

    embeddings = model.encode(
        texts,
        batch_size=32,
        show_progress_bar=True,
        normalize_embeddings=True,
    )
    # Save embeddings to disk
    np.save(embedding_file, embeddings)
    print(f"Embeddings saved to '{embedding_file}'.")

print(f"Embeddings shape: {embeddings.shape}")

# Attach embeddings back to chunk dictionaries
for i, chunk in enumerate(chunks):
    chunk["embedding"] = embeddings[i]

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

No cached embeddings found. Computing with BGE-M3...


Batches:   0%|          | 0/114 [00:00<?, ?it/s]

Embeddings saved to '/kaggle/working/corpus_embeddings.npy'.
Embeddings shape: (3633, 1024)


## 

## Step 5 — Autoencoder: reduce 1024 → 64 dimensions

The VQC has 6 qubits, so it needs exactly 6 numbers per chunk. Autoencoder finds the 6 directions of greatest variance in the 1024-dim embeddings and projects every vector onto just those 6.

**Critical rule**: Autoencoder is fit **once**, on the corpus, and the exact same fitted transform is reused for queries later — never refit on queries. Refitting would put queries and documents in two different, incomparable 6-D coordinate systems.

We also check `explained_variance_ratio_` — this tells us how much of the original signal survives this aggressive 256x compression. (In our own run, this came out to ~11.8% total variance retained — a real, worth-reporting limitation, not a bug.)


In [7]:
# import torch.nn as nn
# import torch

# class Autoencoder(nn.Module):
#     def __init__(self, input_dim=1024, bottleneck=64):
#         super().__init__()
#         self.encoder = nn.Sequential(
#             nn.Linear(input_dim, 256), nn.ReLU(),
#             nn.Linear(256, 128), nn.ReLU(),
#             nn.Linear(128, bottleneck)
#         )
#         self.decoder = nn.Sequential(
#             nn.Linear(bottleneck, 128), nn.ReLU(),
#             nn.Linear(128, 256), nn.ReLU(),
#             nn.Linear(256, input_dim)
#         )
#     def forward(self, x):
#         z = self.encoder(x)
#         return self.decoder(z), z

# BOTTLENECK_DIM = 64  # = 2^6, matches the 6-qubit amplitude encoding exactly

# ae = Autoencoder(input_dim=1024, bottleneck=BOTTLENECK_DIM).double()
# ae_optimizer = torch.optim.Adam(ae.parameters(), lr=1e-3)

# X = torch.tensor(embeddings, dtype=torch.float64)
# n_ae_epochs = 50
# for epoch in range(n_ae_epochs):
#     ae_optimizer.zero_grad()
#     recon, _ = ae(X)
#     loss = nn.functional.mse_loss(recon, X)
#     loss.backward()
#     ae_optimizer.step()
#     if (epoch + 1) % 10 == 0:
#         print(f"AE epoch {epoch+1}/{n_ae_epochs}, recon MSE: {loss.item():.4f}")

# with torch.no_grad():
#     _, reduced_embeddings = ae(X)
# reduced_embeddings = reduced_embeddings.numpy()
# print(f"Reduced shape: {reduced_embeddings.shape}")

## Step 6 — No scaling needed for Autoencoder

The VQC uses each of the 4 PCA numbers as a **rotation angle** for a qubit. Since rotation angles are periodic (2π = back to start), raw PCA values are rescaled into roughly the `-π` to `π` range using the largest absolute value seen across the whole corpus (`max_scale`).

We also embed **queries** here for the first time (BGE-M3 → the *same* fitted PCA → scale), and cache both chunk and query scaled vectors in dictionaries keyed by ID. This means the expensive BGE-M3/PCA work only happens once — the training loop (and later, retrieval) just does fast dictionary lookups instead of recomputing embeddings every time.


In [8]:
import torch

# No angle-scaling needed for amplitude encoding — AmplitudeEmbedding normalizes internally.
chunk_id_to_vec64 = {}
for i, chunk in enumerate(chunks):
    chunk_id_to_vec64[chunk["chunk_id"]] = embeddings[i]

query_texts = [q["text"] for q in queries]
query_embeddings = model.encode(query_texts, normalize_embeddings=True)
# with torch.no_grad():
#     query_reduced = ae.encoder(torch.tensor(query_embeddings, dtype=torch.float64)).numpy()

query_id_to_vec64 = {}
for i, q in enumerate(queries):
    query_id_to_vec64[q["_id"]] = query_embeddings[i]

print(f"Cached {len(chunk_id_to_vec64)} chunk vectors and {len(query_id_to_vec64)} query vectors (64-D each)")

Cached 3633 chunk vectors and 3237 query vectors (64-D each)


## Step 7 — Define the 6-Qubit Variational Quantum Circuit (VQC)

Picture 6 dials (qubits), each able to point in any direction. The circuit does three things, in order:

1. **Encoding** (`encode_data`): each of the 6 autoencoder numbers turns one dial (`qml.RY` = rotate qubit `i` by angle `x[i]`). One number, one dial — no relationship between them yet.
2. **Variational layer** (`variational_layer`): the dials get entangled (linked together) and each gets an extra nudge, controlled by trainable `weights`. This is the only part that changes during training.
3. **Measurement** (`qml.expval(qml.PauliZ(i))`): read the final position of each dial as a plain number between -1 and +1. Four numbers in, four numbers out — this is the "quantum embedding."

`interface="torch"` makes the circuit differentiable via PyTorch's autograd, so gradients can flow through it during training exactly like any normal PyTorch layer.


In [9]:
import pennylane as qml

n_qubits = 10         # 2^10 = 1024
n_layers = 2
dev = qml.device("default.qubit", wires=n_qubits)

def encode_data(x):
    qml.AmplitudeEmbedding(features=x, wires=range(n_qubits), normalize=True, pad_with=0.0)

def variational_layer(weights):
    qml.StronglyEntanglingLayers(weights, wires=range(n_qubits))

@qml.qnode(dev, interface="torch")
def quantum_circuit(x, weights):
    encode_data(x)
    variational_layer(weights)
    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

print("VQC (amplitude encoding, 10 qubits) defined.")

VQC (amplitude encoding, 10 qubits) defined.


## Step 8 — Build training triplets from qrels

The VQC has no built-in notion of "relevant" vs "irrelevant" — its weights start random. Training needs concrete examples to check itself against: for a given question, which document is *actually* correct, and which one (almost certainly) isn't.

Each **triplet** = one (question, correct document, random wrong document) bundle:

1. Pick a random question that has at least one known correct answer (`query_ids_with_pos`)
2. Pick one of its confirmed-relevant documents → the **positive**
3. Pick a random document from the whole corpus, making sure it isn't secretly a correct answer → the **negative**

2000 of these bundles form the training dataset — conceptually the same idea as a labeled dataset for any ML model, just shaped as groups of three instead of (input, label) pairs, since the goal is "pull these two together, push that one apart" rather than classification.


In [10]:
import random

def build_triplets(query_ids_with_pos, query_to_pos_docs, query_id_to_text,
                    doc_id_to_chunk, all_doc_ids, num_triplets=2000):
    triplets = []
    for _ in range(num_triplets):
        qid = random.choice(query_ids_with_pos)
        pos_doc_id = random.choice(query_to_pos_docs[qid])

        neg_doc_id = random.choice(all_doc_ids)
        while neg_doc_id in query_to_pos_docs[qid]:
            neg_doc_id = random.choice(all_doc_ids)

        if qid in query_id_to_text and pos_doc_id in doc_id_to_chunk and neg_doc_id in doc_id_to_chunk:
            triplets.append({
                "query_id": qid,
                "query_text": query_id_to_text[qid],
                "pos_chunk": doc_id_to_chunk[pos_doc_id],
                "neg_chunk": doc_id_to_chunk[neg_doc_id]
            })

    return triplets

triplets = build_triplets(train_qids, query_to_pos_docs, query_id_to_text,
                           doc_id_to_chunk, all_doc_ids, num_triplets=2000)
print(f"Built {len(triplets)} triplets")
print(triplets[0]["query_text"])
print(triplets[0]["pos_chunk"]["text"][:100])
print(triplets[0]["neg_chunk"]["text"][:100])

Built 2000 triplets
natural toxins
Selective induction of apoptosis of human oral cancer cell lines by avocado extracts via a ROS-media
Intake of Fiber and Nuts during Adolescence and Incidence of Proliferative Benign Breast Disease. Ob


## Step 9 — Initialize trainable weights and the optimizer

`qml.StronglyEntanglingLayers.shape(...)` doesn't compute anything about your data — it just tells you how many trainable numbers you need for 4 qubits across 2 layers (shape `(2, 4, 3)`). Those numbers are then randomly initialized as a **PyTorch tensor with `requires_grad=True`**, so PyTorch tracks how changing each of them affects the loss later.

`optimizer = torch.optim.Adam([weights], lr=0.01)` creates the mechanism that will actually apply the nudges to `weights` once training tells it which direction to move — `lr` (learning rate) controls how big each nudge is.


In [11]:
import os
import pennylane as qml
import torch

vqc_file = "/kaggle/working/vqc_weights.pt"
weight_shape = qml.StronglyEntanglingLayers.shape(
    n_layers=n_layers, n_wires=n_qubits
)

if os.path.exists(vqc_file):
    print("Loading trained VQC weights...")
    weights = torch.load(vqc_file)
    weights.requires_grad_(True)
    weights_loaded_from_disk = True
else:
    print("Initializing fresh random weights...")
    weights = torch.tensor(
        np.random.uniform(0, 2 * np.pi, size=weight_shape),
        dtype=torch.float64,
        requires_grad=True,
    )
    weights_loaded_from_disk = False

optimizer = torch.optim.Adam([weights], lr=0.01)

Initializing fresh random weights...


In [12]:
import torch

sample_chunk_ids = list(chunk_id_to_vec64.keys())[:5]

print("Checking untrained circuit outputs for 5 sample documents:\n")
outputs = []
for cid in sample_chunk_ids:
    vec = torch.tensor(chunk_id_to_vec64[cid], dtype=torch.float64)
    with torch.no_grad():
        out = torch.stack(quantum_circuit(vec, weights)).numpy()
    outputs.append(out)
    print(f"{cid}: circuit_output={out.round(4)}")

outputs = np.array(outputs)
print(f"\nStd dev across these 5 outputs, per dimension: {outputs.std(axis=0).round(4)}")

Checking untrained circuit outputs for 5 sample documents:

MED-10: circuit_output=[-0.0365  0.0357 -0.0157  0.0069  0.012  -0.0667  0.0079  0.0693 -0.0021
 -0.0409]
MED-14: circuit_output=[-0.0617  0.0196 -0.0224 -0.0043  0.0399 -0.0377  0.034   0.0533 -0.0096
 -0.0209]
MED-118: circuit_output=[ 0.0127  0.0412  0.0216  0.0252 -0.0119  0.0166 -0.0622  0.0724  0.0142
 -0.0057]
MED-301: circuit_output=[ 0.0065 -0.0386  0.0274  0.04   -0.0033  0.0279  0.0008  0.0322 -0.0795
  0.0015]
MED-306: circuit_output=[ 0.0215 -0.0008  0.013   0.0442  0.0033  0.0094  0.001  -0.0128 -0.0318
  0.0051]

Std dev across these 5 outputs, per dimension: [0.0321 0.029  0.0201 0.0187 0.0178 0.0361 0.0317 0.0313 0.0324 0.0169]


## Step 10 — Triplet loss

For one triplet, `dist_pos` = distance between the question's quantum embedding and its correct answer's (want this **small**). `dist_neg` = distance to the wrong answer's (want this **large**). If `dist_pos` is already smaller than `dist_neg` by at least `margin`, the loss clips to exactly 0 — nothing to fix. Otherwise, the loss is a positive number that tells the optimizer to adjust the weights.


In [13]:
# Replace Cell 88 code with:
import torch.nn.functional as F

def triplet_loss(anchor_emb, pos_emb, neg_emb, margin=0.2):
    # Compute cosine similarity
    sim_pos = F.cosine_similarity(anchor_emb.unsqueeze(0), pos_emb.unsqueeze(0))
    sim_neg = F.cosine_similarity(anchor_emb.unsqueeze(0), neg_emb.unsqueeze(0))
    
    # Cosine distance = 1 - similarity (we want dist_pos < dist_neg)
    dist_pos = 1.0 - sim_pos
    dist_neg = 1.0 - sim_neg
    
    loss = torch.clamp(dist_pos - dist_neg + margin, min=0)
    return loss

print("Cosine Triplet Loss function defined.")

Cosine Triplet Loss function defined.


## Step 11 — Train the VQC

For every triplet: look up its 3 pre-cached scaled vectors → run all three through the *same* dials using the *current* weights → compute the loss → let PyTorch work out which direction each weight should move (`loss.backward()`) → apply the nudge (`optimizer.step()`). One full pass through all 2000 triplets = one epoch; we repeat this 20 times.

**Watch the printed `avg loss` per epoch** — it should generally trend downward as training proceeds. A loss that plateaus well above 0 (as opposed to going to ~0) indicates the circuit only partially learned to separate relevant from irrelevant — worth reporting honestly rather than treated as a bug.

⚠️ This runs the circuit `2000 × 3 × 20 = 120,000` times total — expect this cell to take a while.


In [14]:
n_epochs = 20

if weights_loaded_from_disk:
    print("Skipping training loop — using pre-trained weights loaded from disk.")
else:
    print("Training VQC from scratch...")
    for epoch in range(n_epochs):
        total_loss = 0.0
        for triplet in triplets:
            optimizer.zero_grad()

            anchor_vec = torch.tensor(
                query_id_to_vec64[triplet["query_id"]],
                dtype=torch.float64,
            )
            pos_vec = torch.tensor(
                chunk_id_to_vec64[triplet["pos_chunk"]["chunk_id"]],
                dtype=torch.float64,
            )
            neg_vec = torch.tensor(
                chunk_id_to_vec64[triplet["neg_chunk"]["chunk_id"]],
                dtype=torch.float64,
            )

            anchor_emb = torch.stack(quantum_circuit(anchor_vec, weights))
            pos_emb = torch.stack(quantum_circuit(pos_vec, weights))
            neg_emb = torch.stack(quantum_circuit(neg_vec, weights))

            loss = triplet_loss(anchor_emb, pos_emb, neg_emb)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(
            f"Epoch {epoch + 1}/{n_epochs}, avg loss: {total_loss / len(triplets):.4f}"
        )

    # Save newly trained weights immediately after training finishes
    torch.save(weights, vqc_file)
    print(f"Trained VQC weights saved to {vqc_file}")

Training VQC from scratch...
Epoch 1/20, avg loss: 0.2175
Epoch 2/20, avg loss: 0.1982
Epoch 3/20, avg loss: 0.1907
Epoch 4/20, avg loss: 0.1937
Epoch 5/20, avg loss: 0.1843
Epoch 6/20, avg loss: 0.1960
Epoch 7/20, avg loss: 0.1999
Epoch 8/20, avg loss: 0.1951
Epoch 9/20, avg loss: 0.1932
Epoch 10/20, avg loss: 0.1936
Epoch 11/20, avg loss: 0.1957
Epoch 12/20, avg loss: 0.1967
Epoch 13/20, avg loss: 0.1948
Epoch 14/20, avg loss: 0.1972
Epoch 15/20, avg loss: 0.1932
Epoch 16/20, avg loss: 0.1916
Epoch 17/20, avg loss: 0.1896
Epoch 18/20, avg loss: 0.1932
Epoch 19/20, avg loss: 0.1925
Epoch 20/20, avg loss: 0.1986
Trained VQC weights saved to /kaggle/working/vqc_weights.pt


## Step 12 — Save the trained weights

Kaggle sessions lose everything in memory once closed. Save the trained VQC weights (and re-confirm the PCA model is saved) to `/kaggle/working/` — then **click "Save Version" and wait for it to finish** before closing the session, since that's what actually persists these files on Kaggle's servers.


In [15]:
torch.save(weights, "/kaggle/working/vqc_weights.pt")
print("VQC weights saved!")

# torch.save(ae.state_dict(), "/kaggle/working/ae_weights.pt")
# print("Autoencoder weights saved!")

# Sanity check: reload and confirm it matches
weights_check = torch.load("/kaggle/working/vqc_weights.pt")
print(weights_check.shape)
print(torch.equal(weights, weights_check))  # should print True

VQC weights saved!
torch.Size([2, 10, 3])
True


## Step 13 — Generate quantum embeddings for all documents and build the FAISS index

Run every chunk's cached, scaled PCA vector through the now-**trained** VQC once, producing its final quantum embedding. `torch.no_grad()` is used since we're only *using* the trained circuit now, not training it further — this is faster and uses less memory.

`IndexFlatIP` (inner product) on L2-normalized vectors is equivalent to cosine similarity — "Flat" means exact brute-force search, which is fast enough at this corpus size (3,633 documents, 4 dimensions) that no approximate index is needed.


In [16]:
import faiss

quantum_embeddings_list = []
chunk_ids_ordered = []

with torch.no_grad():
    for chunk in chunks:
        cid = chunk["chunk_id"]
        vec64 = torch.tensor(chunk_id_to_vec64[cid], dtype=torch.float64)
        q_emb = quantum_circuit(vec64, weights)
        q_emb = torch.stack(q_emb).numpy()
        quantum_embeddings_list.append(q_emb)
        chunk_ids_ordered.append(cid)

quantum_embeddings_matrix = np.array(quantum_embeddings_list).astype("float32")
faiss.normalize_L2(quantum_embeddings_matrix)
print(f"Quantum embeddings matrix shape: {quantum_embeddings_matrix.shape}")

quantum_dimension = quantum_embeddings_matrix.shape[1]
quantum_index = faiss.IndexFlatIP(quantum_dimension)
quantum_index.add(quantum_embeddings_matrix)
print(f"Total vectors in quantum FAISS index: {quantum_index.ntotal}")

Quantum embeddings matrix shape: (3633, 10)
Total vectors in quantum FAISS index: 3633


## Step 14 — Quantum search function

A real question must go through the **identical** pipeline documents went through: BGE-M3 → the same fitted PCA → the same scaling → the same trained VQC. Only then is it comparable to what's stored in the FAISS index. FAISS returns row indices, which get mapped back to actual `chunk_id`s via `chunk_ids_ordered`.


In [17]:
def quantum_search(query_text, top_k=5):
    query_emb = model.encode([query_text], normalize_embeddings=True)[0]
    query_vec_tensor = torch.tensor(query_emb, dtype=torch.float64)
    # with torch.no_grad():
    #     query_vec64 = ae.encoder(torch.tensor(query_emb, dtype=torch.float64).unsqueeze(0)).numpy()[0]
    # query_vec64_tensor = torch.tensor(query_vec64, dtype=torch.float64)

    with torch.no_grad():
        q_emb = quantum_circuit(query_vec_tensor, weights)
        q_emb = torch.stack(q_emb).numpy().astype("float32").reshape(1, -1)

    faiss.normalize_L2(q_emb)
    similarities, indices = quantum_index.search(q_emb, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        chunk_id = chunk_ids_ordered[idx]
        chunk = doc_id_to_chunk[chunk_id]
        results.append({
            "rank": rank + 1, "chunk_id": chunk_id,
            "similarity": float(similarities[0][rank]),
            "text_preview": chunk["text"][:150]
        })
    return results

## Step 15 — Classical RAG baseline (for a fair comparison)

This is the missing piece needed to interpret the quantum results: a **classical** pipeline that uses the exact same documents, same BGE-M3 model, same chunking, and same cosine-similarity search — just **without** PCA or the VQC. This isolates the effect of the quantum embedding step specifically, rather than comparing two systems that differ in more than one way.


In [18]:
classical_dimension = embeddings.shape[1]  # 1024
classical_index = faiss.IndexFlatIP(classical_dimension)

classical_embeddings = embeddings.astype("float32").copy()
faiss.normalize_L2(classical_embeddings)
classical_index.add(classical_embeddings)

print(f"Classical FAISS index size: {classical_index.ntotal}")

def classical_search(query_text, top_k=5):
    query_emb = model.encode([query_text], normalize_embeddings=True).astype("float32")
    faiss.normalize_L2(query_emb)
    similarities, indices = classical_index.search(query_emb, top_k)
    return [chunks[idx]["chunk_id"] for idx in indices[0]]

# Quick smoke test
print(classical_search(queries[0]["text"], top_k=3))

Classical FAISS index size: 3633
['MED-2434', 'MED-2439', 'MED-5341']


In [19]:
# ae_64d_dimension = BOTTLENECK_DIM
# ae_64d_index = faiss.IndexFlatIP(ae_64d_dimension)

# with torch.no_grad():
#     ae_64d_embeddings = ae.encoder(torch.tensor(embeddings, dtype=torch.float64)).numpy().astype("float32")

# faiss.normalize_L2(ae_64d_embeddings)
# ae_64d_index.add(ae_64d_embeddings)

# def classical_64d_search(query_text, top_k=5):
#     query_emb = model.encode([query_text], normalize_embeddings=True)[0]
#     with torch.no_grad():
#         q_64d = ae.encoder(torch.tensor(query_emb, dtype=torch.float64).unsqueeze(0)).numpy().astype("float32")
#     faiss.normalize_L2(q_64d)
#     similarities, indices = ae_64d_index.search(q_64d, top_k)
#     return [chunks[idx]["chunk_id"] for idx in indices[0]]

## Step 16 — Evaluation: Recall@5, classical vs quantum, on the SAME fixed test set

**Recall@5**: out of a set of test questions, for what fraction did at least one of the top-5 retrieved documents match a document qrels confirms is actually relevant?

A fixed random seed (`random.seed(42)`) locks in one specific sample of 100 test questions (all guaranteed to have at least one known-correct answer, via `query_ids_with_pos`), so both pipelines are evaluated on **exactly the same questions** — this is what makes the final comparison valid rather than comparing two different random samples.


In [20]:
def evaluate_recall_at_k(search_fn, test_qids, k=5, is_quantum=False):
    hits = 0
    for qid in test_qids:
        query_text = query_id_to_text[qid]
        actual_relevant = set(query_to_pos_docs[qid])

        if is_quantum:
            results = search_fn(query_text, top_k=k)
            retrieved_ids = set(r["chunk_id"] for r in results)
        else:
            retrieved_ids = set(search_fn(query_text, top_k=k))

        if len(actual_relevant & retrieved_ids) > 0:
            hits += 1

    return hits / len(test_qids)

# test_qids comes from Step 3.5 — no reseeding/resampling here
classical_1024_recall = evaluate_recall_at_k(classical_search, test_qids, k=5, is_quantum=False)
# classical_64d_recall = evaluate_recall_at_k(classical_64d_search, test_qids, k=5, is_quantum=False)
quantum_64d_recall = evaluate_recall_at_k(quantum_search, test_qids, k=5, is_quantum=True)

print(f"Classical (1024-D)      Recall@5: {classical_1024_recall:.2%}")
# print(f"Classical (64-D AE)     Recall@5: {classical_64d_recall:.2%}")
print(f"Quantum   (10q ampl-enc) Recall@5: {quantum_64d_recall:.2%}")

Classical (1024-D)      Recall@5: 62.00%
Quantum   (10q ampl-enc) Recall@5: 9.00%


## Interpreting the result

Some context worth writing into your report, regardless of what numbers come out:

- **This measures ONE thing**: whether adding PCA + a trained 4-qubit VQC on top of BGE-M3 embeddings helps, hurts, or doesn't change retrieval quality on NFCorpus, using Recall@5.
- Two upstream factors are worth reporting alongside this result if quantum underperforms: (1) `pca.explained_variance_ratio_.sum()` — how much signal survived the 1024→4 compression, and (2) how far the final training loss got from 0 — how well the VQC learned to separate relevant from irrelevant during training. Both bound how good the quantum pipeline could possibly be, independent of the retrieval evaluation itself.
- If you want additional evidence beyond Recall@5, the same `evaluate_recall_at_k` function works for other values of `k` (e.g. `k=1`, `k=10`), and you could also try a middle baseline — classical BGE-M3 embeddings passed through PCA only (no VQC) — to isolate how much of any gap is due to PCA's compression alone versus the VQC's transformation.
